In [ ]:
import pandas as pd
import numpy as np
import requests
import pyodbc
import arcpy
from arcgis.features import FeatureLayer
from utils import *
# This is using Andy's Census API KEy
census_api_key = '9a73d08c296b844e58f1c70bd19c831826da5cbf'

# Need to define datatypes so that FIPS code doesn't get cast as int and drop leading 0s
dtypes = {
    'YEAR' : str,
    'STATE': str,
    'GEOGRAPHY': str,
    'GEOID': str,
    'TRPAID':str,
    'NEIGHBORHOOD': str
}

#Get tahoe basin census geometryies
 
service_url = 'https://maps.trpa.org/server/rest/services/Demographics/MapServer/27'

feature_layer = FeatureLayer(service_url)
tahoe_geometry_fields = ['YEAR', 'STATE', 'GEOGRAPHY', 'GEOID', 'TRPAID', 'NEIGHBORHOOD']
query_result = feature_layer.query(out_fields=",".join(tahoe_geometry_fields))
# Convert the query result to a list of dictionaries
feature_list = query_result.features

# Create a pandas DataFrame from the list of dictionaries
tahoe_geometry = pd.DataFrame([feature.attributes for feature in feature_list])

# Get list of current variables

In [ ]:
current_variables = get_existing_variables(year=2024,dataset='acs/acs5',feature_layer_url='https://maps.trpa.org/server/rest/services/Demographics/MapServer/28')

# Download new variables

In [ ]:
session = make_session()
year = 2024
checkpoint_dir = 'checkpoints'
new_data = census_download_wrapper_checkpointed(current_variables,year,checkpoint_dir,
                                                final_output_csv=f'census_data_{year}.csv',
                                                tahoe_geometry=tahoe_geometry,
                                                session=session,
                                                census_geom_year=2020,
                                                census_api_key=census_api_key)